## 📦 Step 1: Check GPU & Install Dependencies

In [ ]:
# Check GPU availability
!nvidia-smi

# Install build dependencies
!apt-get update -qq
!apt-get install -y -qq build-essential cmake git

# Check CUDA version
!nvcc --version

print("\n✅ GPU and dependencies ready!")

## 📥 Step 2: Download ZIP from Google Drive & Extract

In [ ]:
import os

# Clean up any previous installation
if os.path.exists('/content/bitrecover'):
    !rm -rf /content/bitrecover

# Navigate to colab content dir
%cd /content

# Install gdown for Google Drive downloads
!pip install -q gdown
import gdown

# ═══════════════════════════════════════════════════════════════
# IMPORTANT: To avoid the Google Drive 'file limit error', do NOT use a folder link.
# 1. Compress your 'bitrecover' folder into 'bitrecover.zip' on your computer.
# 2. Upload 'bitrecover.zip' to Google Drive.
# 3. Get the shareable link for 'bitrecover.zip' and paste the FILE ID below.
# ═══════════════════════════════════════════════════════════════
zip_file_id = "1cqjim3LuGFOQ9kNckXvjmjNaBO_F26q0"  # CHANGE THIS TO YOUR ZIP FILE ID!

try:
    print("📥 Downloading bitrecover.zip...")
    gdown.download(f"https://drive.google.com/uc?id={zip_file_id}", "bitrecover.zip", quiet=False)
    
    print("📦 Extracting bitrecover.zip...")
    !unzip -q bitrecover.zip -d /content/
    !rm bitrecover.zip
    print("✅ Project downloaded and extracted successfully!")
except Exception as e:
    print(f"❌ ERROR: Failed to download or extract ZIP: {e}")

%cd /content/bitrecover

# ═══════════════════════════════════════════════════════════════
# IMPORTANT: Update these Google Drive file IDs with your config & address!
# ═══════════════════════════════════════════════════════════════
config_file_id = "1t3Px1xZtNK2ntntEpK0lZyyBCGRjLz69"  # CHANGE THIS!
address_file_id = "1loSt1inO-Pai0q2-tkcsGNN0J0DpLpbF"  # CHANGE THIS!

# Download and replace config.json
try:
    os.makedirs('config', exist_ok=True)
    gdown.download(f"https://drive.google.com/uc?id={config_file_id}", "config/config.json", quiet=False)
    print("✅ Downloaded your config.json from Google Drive")
except Exception as e:
    print(f"⚠️ Error downloading config: {e}")

# Download and replace address.txt
try:
    gdown.download(f"https://drive.google.com/uc?id={address_file_id}", "address.txt", quiet=False)
    print("✅ Downloaded your address.txt from Google Drive")
except Exception as e:
    print(f"⚠️ Error downloading address.txt: {e}")

print("\n✅ All configuration files ready!")


## 🔨 Step 3: Build Bitrecover

In [ ]:
# Create build directory and compile
!mkdir -p build
%cd build
!cmake ..
!make -j$(nproc)
%cd ..

# Verify binaries exist
import os
if os.path.exists('build/bin/bitrecover'):
    print("\n✅ Build complete! Bitrecover is ready to run.")
else:
    print("\n❌ Build failed! Check errors above.")

## 🚀 Step 4: Run Bitrecover with INSTANT Auto-Download

In [ ]:
# List available GPUs
!./build/bin/bitrecover --list-devices

# Install watchdog for INSTANT file detection (no polling!)
!pip install -q watchdog

import os
import time
import threading
from google.colab import files
from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler

# Flag to track if we've downloaded
downloaded = False

class SuccessFileHandler(FileSystemEventHandler):
    """Handles file creation events - INSTANT detection!"""

    def on_created(self, event):
        global downloaded

        # Check if Success.txt was created
        if event.src_path.endswith('Success.txt') and not downloaded:
            # Wait a moment to ensure file is fully written
            time.sleep(1)

            # Read and display the match
            try:
                with open('Success.txt', 'r') as f:
                    content = f.read()

                print("\n" + "="*65)
                print("🎉🎉🎉 MATCH FOUND! 🎉🎉🎉")
                print("="*65)
                print(content)
                print("="*65)

                # Auto-download immediately
                print("\n📥 Auto-downloading Success.txt...")
                files.download('Success.txt')
                print("✅ Success.txt downloaded to your computer!\n")

                downloaded = True
            except Exception as e:
                print(f"Error downloading: {e}")

    def on_modified(self, event):
        """Also trigger on file modification (when content is added)"""
        if event.src_path.endswith('Success.txt') and not downloaded:
            self.on_created(event)

# Set up file system watcher
event_handler = SuccessFileHandler()
observer = Observer()
observer.schedule(event_handler, path='.', recursive=False)
observer.start()

print("🔍 INSTANT file monitoring active (using watchdog)!\n")
print("⚡ Zero polling overhead - Success.txt will be detected immediately!\n")

# Run bitrecover

print("********** Starting Bitrecover - PRESS STOP TO HALT **********")
print()
print("📊 Live Stats:")
print("   - GPU Speed (MKeys/s)")
print("   - Keys Processed")
print("   - GPU Utilization")
print()
print("💡 Success.txt will auto-download INSTANTLY when found!")
print("⏹️  Press the STOP button to halt the search")
print()
print("═"*65)
print()

try:
    # Run bitrecover - will show live stats
    !./build/bin/bitrecover --config config/config.json
except KeyboardInterrupt:
    print("\n⏹️  Search stopped by user")
finally:
    # Stop file watcher
    observer.stop()
    observer.join()

print("\n✅ Bitrecover session complete!")